In [10]:
import re
import pandas as pd

In [11]:
import pandas as pd
import re


def process_eko_transactions():

    data = pd.read_excel("data/eko_transaction.xlsx", dtype={"Number": str})
    cards = pd.read_excel("data/cards.xlsx", dtype={"Number": str})

    # -------------------------
    # CLEAN MATERIAL
    # -------------------------
    data["Material"] = (
        data["Material"]
        .astype(str)
        .str.replace(r'^\d+\s*', '', regex=True)
        .str.strip()
    )

    # -------------------------
    # CLEAN VEHICLE NAME
    # -------------------------
    data["Name"] = (
        data["Name"]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )

    # -------------------------
    # SELECT COLUMNS
    # -------------------------
    data = data[
        [
            "Plant",
            "Name",
            "Number",
            "Material",
            "Date",
            "Bill.qty",
            "Bill.qty.1",
            "FinPr",
            "Tot Amount",
            "Auth.time",
        ]
    ]

    # -------------------------
    # CLEAN CARD NUMBERS
    # -------------------------
    data["Number"] = (
    data["Number"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
    )

    cards["Number"] = (
        cards["Number"]
        .astype(str)
        .str.replace(r"\D", "", regex=True)
    )

    # премахване на празни номера
    data = data[data["Number"].str.len() > 0]

    # -------------------------
    # REMOVE NON TRANSACTIONS
    # -------------------------
    data["Bill.qty"] = pd.to_numeric(data["Bill.qty"], errors="coerce")
    data = data[data["Bill.qty"] > 0]

    # -------------------------
    # MERGE COMPANY
    # -------------------------
    data = data.merge(
        cards[["Number", "Company"]],
        on="Number",
        how="left"
    )

    # -------------------------
    # WRITE EXCEL REPORT
    # -------------------------
    output_file = "result/eko_transactions.xlsx"

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

        if data.empty:
            pd.DataFrame({"Info": ["No transactions found"]}).to_excel(
                writer,
                sheet_name="Report",
                index=False
            )
        else:
            for company, df in data.groupby("Company", dropna=False):

                company_name = "Unknown" if pd.isna(company) else str(company)

                sheet_name = re.sub(r'[\\/*?:\[\]]', '', company_name)[:31]

                if sheet_name == "":
                    sheet_name = "Unknown"

                df.to_excel(writer, sheet_name=sheet_name, index=False)

    return data

In [12]:
data = process_eko_transactions()